# AGROBRIDGE — Pomegranate Disease Model Training (EfficientNet-B0)

**Objective**: Fine-tune a lightweight, high-performance EfficientNet-B0 backbone for 5-class pomegranate disease detection matching the AGROBRIDGE `INFERENCE_CONTRACT`.

> **Runtime Requirement**: Cloud GPU (NVIDIA T4 / P100 on Kaggle or Google Colab). Saves trained checkpoints outside the repository.

In [ ]:
import os
import time
import copy
import json
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import f1_score, accuracy_score

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active training device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CLASS_LABELS = ["Healthy", "Anthracnose", "Bacterial_Blight", "Alternaria", "Cercospora"]
LABEL2IDX = {label: idx for idx, label in enumerate(CLASS_LABELS)}
IDX2LABEL = {idx: label for idx, label in enumerate(CLASS_LABELS)}

MODEL_SAVE_DIR = Path("/kaggle/working/models") if Path("/kaggle").exists() else Path("./saved_models")
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Dataset & Robust Augmentation Pipeline

In [ ]:
class PomegranateDataset(Dataset):
    def __init__(self, manifest, transform=None, is_mock=False):
        self.manifest = manifest
        self.transform = transform
        self.is_mock = is_mock

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        label_name = row["label"]
        label_idx = LABEL2IDX[label_name]

        if self.is_mock or not os.path.exists(str(row["path"])):
            # Synthetic placeholder image for dry-run verification
            image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
        else:
            image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label_idx, dtype=torch.long)

# Agricultural field augmentations (lighting changes, orientation, leaf angles)
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 2. Load Manifests and Configure DataLoaders

In [ ]:
# Load manifests if available
manifest_dir = Path('/kaggle/working/pomegranate_prepared') if Path('/kaggle').exists() else Path('./pomegranate_prepared')
train_csv = manifest_dir / 'train_manifest.csv'
val_csv = manifest_dir / 'val_manifest.csv'

is_mock = False
if train_csv.exists() and val_csv.exists():
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
else:
    print("Manifests not found in local dir, running dry-run dataset mock.")
    is_mock = True
    records = []
    for cls in CLASS_LABELS:
        for _ in range(50):
            records.append({"path": "mock.jpg", "label": cls})
    train_df = pd.DataFrame(records)
    val_df = pd.DataFrame(records[:50])

train_dataset = PomegranateDataset(train_df, transform=train_transforms, is_mock=is_mock)
val_dataset = PomegranateDataset(val_df, transform=val_transforms, is_mock=is_mock)

BATCH_SIZE = 32 if device.type == "cuda" else 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2 if device.type == 'cuda' else 0, pin_memory=True if device.type == 'cuda' else False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2 if device.type == 'cuda' else 0, pin_memory=True if device.type == 'cuda' else False)

# Compute class weights to address any imbalances
counts = [train_df["label"].value_counts().get(c, 1) for c in CLASS_LABELS]
weights = torch.tensor([sum(counts) / (len(counts) * c) for c in counts], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
print("Class weights applied to loss:", {c: round(w.item(), 3) for c, w in zip(CLASS_LABELS, weights)})

## 3. Model Architecture (EfficientNet-B0 Transfer Learning)

In [ ]:
# Load pre-trained EfficientNet-B0
weights_enum = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights_enum)

# Custom classifier head
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(num_ftrs, len(CLASS_LABELS))
)
model = model.to(device)
print("EfficientNet-B0 initialized with 5-class classifier head.")

## 4. Two-Phase Training Strategy
- **Phase 1 (Warmup)**: Freeze feature extractor, train classifier head (3 epochs, lr=1e-3).
- **Phase 2 (End-to-End Fine-Tuning)**: Unfreeze top layers, train with Cosine Annealing scheduler (12 epochs, lr=1e-4).

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1

def eval_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1

# ---------------- Phase 1: Warmup Head ----------------
print("=== Phase 1: Classifier Head Warmup ===")
for param in model.features.parameters():
    param.requires_grad = False

optimizer_head = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-2)
WARMUP_EPOCHS = 3
for epoch in range(WARMUP_EPOCHS):
    t0 = time.time()
    t_loss, t_acc, t_f1 = train_epoch(model, train_loader, criterion, optimizer_head)
    v_loss, v_acc, v_f1 = eval_epoch(model, val_loader, criterion)
    print(f"Warmup Epoch {epoch+1}/{WARMUP_EPOCHS} ({time.time()-t0:.1f}s) | Train Loss: {t_loss:.4f} Acc: {t_acc:.3f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.3f} F1: {v_f1:.3f}")

# ---------------- Phase 2: Full Fine-Tuning ----------------
print("\n=== Phase 2: Fine-Tuning Backbone ===")
for param in model.features.parameters():
    param.requires_grad = True

optimizer_fine = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_fine, T_max=12, eta_min=1e-6)

FINE_EPOCHS = 12 if not is_mock else 2
best_val_f1 = 0.0
best_weights = None
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(FINE_EPOCHS):
    t0 = time.time()
    t_loss, t_acc, t_f1 = train_epoch(model, train_loader, criterion, optimizer_fine)
    v_loss, v_acc, v_f1 = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)
    history["val_f1"].append(v_f1)

    print(f"Epoch {epoch+1:02d}/{FINE_EPOCHS:02d} ({time.time()-t0:.1f}s) | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.3f} F1: {v_f1:.3f}")

    if v_f1 >= best_val_f1:
        best_val_f1 = v_f1
        best_weights = copy.deepcopy(model.state_dict())
        print(f"  → New best validation F1: {v_f1:.4f}! Checkpoint saved.")

## 5. Export Best Model Checkpoint

In [ ]:
if best_weights is not None:
    model.load_state_dict(best_weights)

checkpoint_path = MODEL_SAVE_DIR / "pomegranate-fruit-v1.pth"
torch.save({
    "model_architecture": "efficientnet_b0",
    "model_state_dict": model.state_dict(),
    "class_labels": CLASS_LABELS,
    "label_to_index": LABEL2IDX,
    "image_size": 224,
    "best_val_f1": best_val_f1,
    "training_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "contract_version": "pomegranate-fruit-v1"
}, checkpoint_path)

print(f"✓ Checkpoint successfully exported to: {checkpoint_path}")
print(f"File size: {checkpoint_path.stat().st_size / (1024*1024):.2f} MB")